# GX_01 — Ingestion and Data Quality (Phase 1)

**Client question:** *Can we trust the data?*

This notebook loads both source files **exactly as delivered**, checks them, and stores them as **bronze** Delta tables. It only *measures* problems; it does not fix them. Every later notebook reads from these tables, never from the CSVs.

| | |
|---|---|
| **Inputs** | `churn.csv`, `bank_transactions.csv` in the Volume `/Volumes/workspace/churn/raw` |
| **Outputs** | `workspace.churn.bronze_churn`, `workspace.churn.bronze_transactions`, `workspace.churn.quality_report` |
| **Next** | `GX_02_data_engineering` builds silver/gold from bronze |

**Why a bronze layer (medallion architecture).** Raw data is copied once into a reliable table format (Delta: ACID writes, schema enforcement, version history). If a cleaning rule turns out to be wrong in Phase 2, we fix the rule and rebuild from bronze; we never go back to the files or lose the original values.

**Design choices**
- **Explicit schemas** (`src/schemas.py`) instead of `inferSchema`: no extra scan of the data, types are documented, and an unexpected value shows up as a malformed row instead of silently changing a column's type.
- **PERMISSIVE mode + `_corrupt_record`**: rows that do not fit the schema are kept and counted, not dropped.
- **Lineage columns** `_ingested_at` and `_source_file` on every bronze row.
- **One reusable quality function** (`src/quality.py`) runs the same checks on both datasets; results are stacked into one table.
- **Re-runnable**: tables are overwritten, so running the notebook twice gives the same result.

## 0. Setup

In [1]:
# Make the shared helpers in src/ importable (this notebook lives in notebooks/).
import os
import sys

REPO_ROOT = os.path.abspath("..")
if REPO_ROOT not in sys.path:
    sys.path.append(REPO_ROOT)

from pyspark.sql import functions as F

from src.config import (
    BRONZE_CHURN, BRONZE_TRANSACTIONS, CHURN_CSV, EXPECTED_ROWS,
    QUALITY_REPORT, RAW_PATH, TRANSACTIONS_CSV,
)
from src.ingest import META_COLS, read_raw_csv, write_bronze
from src.quality import category_counts, quality_report
from src.schemas import CHURN_SCHEMA, CORRUPT_COL, TRANSACTIONS_SCHEMA

print("Spark version:", spark.version)

ModuleNotFoundError: No module named 'pyspark'

In [ ]:
# Raw files in the Volume (uploaded once in GX_00_setup)
display(dbutils.fs.ls(RAW_PATH))

## 1. Customers — `churn.csv`

**Read options**
- `encoding = ISO-8859-1`: kept from GX_00 as a safe default for European text. Note: the 92 surnames containing `?` (e.g. `H?`, `L?`) already contain a literal `?` in the delivered file — the original characters were lost **before** we received the data, so no encoding can recover them. Surname is not used for analysis, so this has no impact (logged in `docs/decisions.md`).
- No renaming needed: all column names are valid Delta column names.

In [ ]:
raw_churn = read_raw_csv(spark, CHURN_CSV, CHURN_SCHEMA, encoding="ISO-8859-1")
write_bronze(raw_churn, BRONZE_CHURN)

bronze_churn = spark.table(BRONZE_CHURN)
print(f"{BRONZE_CHURN}: {bronze_churn.count():,} rows, {len(bronze_churn.columns)} columns")
bronze_churn.printSchema()

In [ ]:
# Preview without Surname (personal data is not displayed)
display(bronze_churn.drop("Surname", *META_COLS).limit(5))

### 1.1 Quality checks

Valid ranges below are **sanity bounds** (what is physically/plausibly possible), not business rules:
- `CreditScore` 300–850: the common credit-score scale.
- `Age` 18–100: adult bank customers.
- `Tenure` 0–10: the documented range of years with the bank.
- `NumOfProducts` 1–4: every customer holds at least one product.
- `Balance`, `EstimatedSalary` ≥ 0 (upper bound is only there to catch absurd values).

In [ ]:
churn_quality = quality_report(
    bronze_churn,
    dataset="churn",
    expected_rows=EXPECTED_ROWS["churn"],
    key_col="CustomerId",
    ranges={
        "CreditScore": (300, 850),
        "Age": (18, 100),
        "Tenure": (0, 10),
        "NumOfProducts": (1, 4),
        "Balance": (0, 10_000_000),
        "EstimatedSalary": (0, 10_000_000),
    },
    allowed_values={
        "Geography": ["France", "Germany", "Spain"],
        "Gender": ["Male", "Female"],
        "HasCrCard": [0, 1],
        "IsActiveMember": [0, 1],
        "Exited": [0, 1],
    },
    rate_cols=["Exited", "HasCrCard", "IsActiveMember"],
)

# Summary first, then only the rows that need attention (full report is saved in section 3)
display(churn_quality.groupBy("status").count().orderBy("status"))
display(churn_quality.filter("status != 'PASS'"))

In [ ]:
# Category distributions: confirms no typos or unexpected labels
display(category_counts(bronze_churn, ["Geography", "Gender", "NumOfProducts", "HasCrCard", "IsActiveMember", "Exited"]))

In [ ]:
# Numeric ranges and spread
numeric_cols = ["CreditScore", "Age", "Tenure", "Balance", "EstimatedSalary"]
display(bronze_churn.select(numeric_cols).summary("count", "min", "25%", "50%", "75%", "max"))

### 1.2 Oddities worth a closer look

The generic checks pass, but three things look unusual for real bank data. We **measure** them here; Phase 2 decides what to do.

In [ ]:
oddities = spark.sql(f'''
    SELECT
        COUNT_IF(Surname LIKE '%?%')                     AS surnames_with_question_mark,
        COUNT_IF(Balance = 0)                            AS zero_balance_customers,
        ROUND(AVG(CASE WHEN Balance = 0 THEN 1 ELSE 0 END) * 100, 1) AS zero_balance_pct,
        COUNT_IF(CreditScore = 850)                      AS credit_score_at_850,
        COUNT_IF(EstimatedSalary < 1000)                 AS salary_below_1000,
        ROUND(MIN(EstimatedSalary), 2)                   AS min_salary
    FROM {BRONZE_CHURN}
''')
display(oddities)

## 2. Transactions — `bank_transactions.csv`

**Read options**
- `quote='"'`, `escape='"'`: the geolocation field contains a comma inside quotes (`"34.0522 N, -74.006 W"`); without these the row splits into too many columns.
- **Column names → snake_case** (mapping in `src/schemas.py`): Delta does not accept spaces or brackets in column names. Values are unchanged.
- `Timestamp`, `Geolocation` and `PIN Code` stay as **strings** in bronze. Parsing them is an interpretation (Phase 2); `pin_code` is a string so leading zeros would survive.

In [ ]:
raw_transactions = read_raw_csv(spark, TRANSACTIONS_CSV, TRANSACTIONS_SCHEMA, quote='"', escape='"')
write_bronze(raw_transactions, BRONZE_TRANSACTIONS)

bronze_transactions = spark.table(BRONZE_TRANSACTIONS)
print(f"{BRONZE_TRANSACTIONS}: {bronze_transactions.count():,} rows, {len(bronze_transactions.columns)} columns")
bronze_transactions.printSchema()

In [ ]:
# Preview without the PIN code (security data is not displayed)
display(bronze_transactions.drop("pin_code", *META_COLS).limit(5))

### 2.1 Quality checks

In [ ]:
transactions_quality = quality_report(
    bronze_transactions,
    dataset="transactions",
    expected_rows=EXPECTED_ROWS["transactions"],
    key_col="transaction_id",
    ranges={
        "amount": (0, 1_000_000),
        "latency_ms": (0, 1_000),
        "slice_bandwidth_mbps": (0, 10_000),
    },
    allowed_values={
        "transaction_type": ["Deposit", "Withdrawal", "Transfer"],
        "transaction_status": ["Success", "Failed"],
        "device_used": ["Mobile", "Desktop"],
        "network_slice_id": ["Slice1", "Slice2", "Slice3"],
    },
    rate_cols=["fraud_flag"],
)

display(transactions_quality.groupBy("status").count().orderBy("status"))
display(transactions_quality.filter("status != 'PASS'"))

In [ ]:
display(category_counts(bronze_transactions, ["transaction_type", "transaction_status", "device_used", "network_slice_id", "fraud_flag"]))

### 2.2 Domain checks (Spark SQL)

Generic checks cannot tell whether the data makes *sense*. Four questions that decide how much we can use this file later:
1. **Time coverage** — how long a period does it cover?
2. **Geolocation** — are the coordinates real places?
3. **Account network** — do accounts transact repeatedly (needed for graph analysis in Phase 6)?
4. **Link to customers** — can transactions be joined to `churn.csv`?

In [ ]:
# 1. Time coverage
display(spark.sql(f'''
    SELECT
        MIN(TO_TIMESTAMP(timestamp_raw))                         AS first_transaction,
        MAX(TO_TIMESTAMP(timestamp_raw))                         AS last_transaction,
        COUNT_IF(TO_TIMESTAMP(timestamp_raw) IS NULL)            AS unparseable_timestamps,
        COUNT(DISTINCT DATE_TRUNC('MINUTE', TO_TIMESTAMP(timestamp_raw))) AS distinct_minutes
    FROM {BRONZE_TRANSACTIONS}
'''))

In [ ]:
# 2. Geolocation: format, number of distinct latitudes/longitudes, hemisphere letters
display(spark.sql(f'''
    WITH parsed AS (
        SELECT
            geolocation_raw,
            REGEXP_EXTRACT(geolocation_raw, '^(-?[0-9.]+) [NS]', 1)  AS lat_text,
            REGEXP_EXTRACT(geolocation_raw, ', (-?[0-9.]+) [EW]$', 1) AS lon_text,
            RIGHT(geolocation_raw, 1)                                AS lon_letter
        FROM {BRONZE_TRANSACTIONS}
    )
    SELECT
        COUNT(*)                                   AS total_rows,
        COUNT_IF(lat_text = '' OR lon_text = '')   AS badly_formatted,
        COUNT(DISTINCT lat_text)                   AS distinct_latitudes,
        COUNT(DISTINCT lon_text)                   AS distinct_longitudes,
        COUNT(DISTINCT geolocation_raw)            AS distinct_points,
        COUNT_IF(lon_letter = 'W')                 AS labelled_west,
        COUNT_IF(lon_text LIKE '-%')               AS negative_longitude
    FROM parsed
'''))

In [ ]:
# 3. Account network: how often does each account appear (as sender or receiver)?
display(spark.sql(f'''
    WITH appearances AS (
        SELECT sender_account_id AS account_id FROM {BRONZE_TRANSACTIONS}
        UNION ALL
        SELECT receiver_account_id FROM {BRONZE_TRANSACTIONS}
    ),
    per_account AS (
        SELECT account_id, COUNT(*) AS n_transactions FROM appearances GROUP BY account_id
    )
    SELECT n_transactions, COUNT(*) AS n_accounts
    FROM per_account
    GROUP BY n_transactions
    ORDER BY n_transactions
'''))

In [ ]:
# 4. Join key: transaction accounts look like 'ACC12345'; customer IDs are 8-digit numbers.
display(spark.sql(f'''
    WITH tx_accounts AS (
        SELECT sender_account_id AS account_id FROM {BRONZE_TRANSACTIONS}
        UNION
        SELECT receiver_account_id FROM {BRONZE_TRANSACTIONS}
    )
    SELECT
        COUNT(*)                                    AS distinct_tx_accounts,
        COUNT_IF(t.account_id RLIKE '^ACC[0-9]+$')  AS accounts_in_ACC_format,
        COUNT(c.CustomerId)                         AS numeric_part_matches_a_customer
    FROM tx_accounts t
    LEFT JOIN (SELECT DISTINCT CustomerId FROM {BRONZE_CHURN}) c
        ON CAST(REGEXP_EXTRACT(t.account_id, '([0-9]+)', 1) AS BIGINT) = c.CustomerId
'''))

## 3. Combined quality report

Both reports are stacked into one Delta table, so the results are queryable later (and visible in the defence) without re-running this notebook.

In [ ]:
full_report = (
    churn_quality.unionByName(transactions_quality)
    .withColumn("_checked_at", F.current_timestamp())
)
write_bronze(full_report, QUALITY_REPORT)

display(
    spark.table(QUALITY_REPORT)
    .groupBy("dataset")
    .pivot("status", ["PASS", "WARN", "FAIL", "INFO"])
    .count()
    .fillna(0)
)

In [ ]:
# Gate: later notebooks assume bronze is structurally sound. Stop here if any hard check failed.
failed = spark.table(QUALITY_REPORT).filter("status = 'FAIL'")
n_failed = failed.count()
if n_failed > 0:
    display(failed)
    raise ValueError(f"{n_failed} quality check(s) failed - fix ingestion before continuing.")
print("All hard checks passed - bronze tables are ready for GX_02.")

## 4. Findings and hand-over to Phase 2

**Customers (`churn.csv`) — clean and usable.**
- 10,000 rows, 14 columns, 0 malformed rows, 0 nulls, 0 duplicate rows, 0 duplicate `CustomerId`. All values inside their valid ranges and categories.
- Churn rate **20.4%** (2,037 customers) → classes are imbalanced; this drives the metric choice in Phase 4.
- Oddities (not errors, logged for Phase 2):
  - **3,617 customers (36.2%) have a balance of exactly 0** — a real group, not missing data (zero-balance customers churn *less*). → `ZeroBalance` feature.
  - **233 customers sit at exactly 850**, the top of the credit-score scale — looks capped at the source. → keep; note in the defence.
  - **59 customers have an "annual salary" below €1,000** (minimum €11.58); salaries look uniformly spread — likely synthetic. → keep, but salary is a weak/unreliable feature.
  - **92 surnames contain a literal `?`** in the source file. → no action; surname is never used.

**Transactions (`bank_transactions.csv`) — structurally clean, but weak as evidence.**
- 1,000 rows, 0 malformed, 0 nulls, 0 duplicate IDs, all categories valid. Fraud flag rate **48.1%** — far too high for real transactions.
- **Covers one hour only** (17 Jan 2025, 10:01–11:00).
- **Geolocation is not real**: 6 latitudes × 6 longitudes combined at random (36 points), every point labelled `W` even with negative or eastern longitudes. → parse the numeric value in Phase 2; do not use for location insights.
- **Almost no repeat accounts**: 1,971 accounts across 1,000 transactions; 1,943 appear once, 27 twice, 1 three times. → the transaction graph will be fragmented (Phase 6, Part B).
- **No join key to customers** (`ACC…` IDs vs 8-digit customer IDs). → transactions cannot enrich the churn model.

**Hand-over:** silver tables cast/parse types, drop `PIN Code` (privacy), and keep IDs only as keys. Each item above has a line in `docs/decisions.md`.